# UTS 41014 — Track A | TurtleBot3 Waffle Pi (Tutor Run-All)

**Run:** Open this notebook in Google Colab → **Runtime → Run all**. If opened without the GitHub repository files, the first setup cell prompts for `TrackA_Tutor_Ready.zip` (provided alongside this notebook). Alternatively, set `GITHUB_REPO_URL` to your public repository URL to clone it automatically.

**Pipeline:** MuJoCo wheel actuators → wheel-joint odometry and geometric 2D LiDAR + synthetic projected camera marker corners → EKF → A* reference → LQR tracking + LiDAR safety. **Ground truth is used for evaluation only**, not control.

**Important limitations:** The full navigation uses geometric simulated camera marker corners rather than rendered-image ArUco detection. The supplied 3D video is a pose replay, not a new physics simulation. Wheel radius and separation used by the controller are empirically calibrated *effective* parameters, not physical dimensions. Results may differ slightly across MuJoCo versions.

In [ ]:
# 1. Install runtime dependencies and locate project assets
import os, sys, subprocess, pathlib, zipfile
os.environ['MUJOCO_GL'] = 'egl'  # Colab's offscreen renderer; set before importing mujoco
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'mujoco>=3.1,<4', 'numpy', 'scipy', 'matplotlib', 'opencv-contrib-python-headless', 'imageio', 'imageio-ffmpeg'])
GITHUB_REPO_URL = ''  # Optional: e.g. 'https://github.com/yourname/TrackA_Tutor_Ready.git'
ROOT = pathlib.Path.cwd()
def locate():
    for base in [ROOT, ROOT/'TrackA_Tutor_Ready']:
        if (base/'tb3nav'/'config.py').exists() and (base/'trackA_waffle_pi_adapted.xml').exists():
            return base
    return None
project = locate()
if project is None and GITHUB_REPO_URL:
    subprocess.check_call(['git','clone','--depth','1',GITHUB_REPO_URL,'TrackA_Tutor_Ready'])
    project = locate()
if project is None:
    from google.colab import files
    print('Upload TrackA_Tutor_Ready.zip from the supplied project package.')
    uploaded = files.upload()
    zips = [n for n in uploaded if n.lower().endswith('.zip')]
    if not zips: raise FileNotFoundError('No project ZIP uploaded.')
    with zipfile.ZipFile(zips[0]) as z:
        if any('..' in pathlib.PurePosixPath(n).parts or n.startswith('/') for n in z.namelist()):
            raise ValueError('Unsafe ZIP paths')
        z.extractall(ROOT)
    project = locate()
if project is None: raise FileNotFoundError('Expected tb3nav/ and trackA_waffle_pi_adapted.xml')
os.chdir(project)
sys.path.insert(0,str(project))
print('Project directory:',project)

In [ ]:
# 2. Initialise the original successful model and calibrated controller parameters
import mujoco, numpy as np, matplotlib.pyplot as plt
from tb3nav.config import Config
cfg = Config()
cfg.wheel_radius = 0.0150    # empirically effective radius, NOT physical wheel radius
cfg.wheel_sep = 0.2199       # empirically effective track width
cfg.robot_radius = 0.19     # collision-footprint approximation
cfg.v_max = 0.10
cfg.v_ref = 0.08
cfg.omega_max = 0.8
np.random.seed(42)
model_stable = mujoco.MjModel.from_xml_path('trackA_waffle_pi_adapted.xml')
data_stable = mujoco.MjData(model_stable)
assert model_stable.nu == 2, 'Expected two wheel motors'
for n in ['robot','wheel_left','wheel_right']:
    assert mujoco.mj_name2id(model_stable,mujoco.mjtObj.mjOBJ_BODY,n)>=0,n
for n in ['x_joint','y_joint','yaw_joint','left_wheel_joint','right_wheel_joint']:
    assert mujoco.mj_name2id(model_stable,mujoco.mjtObj.mjOBJ_JOINT,n)>=0,n
print('MuJoCo model loaded:',model_stable.nbody,'bodies;',model_stable.njnt,'joints')

## Planar adapter

In [ ]:
import numpy as np
import mujoco

from tb3nav.sim import Measurements
from tb3nav.mujoco_adapter import (
    wheels_from_cmd,
    odom_from_wheels
)
from tb3nav.utils import wrap

class PlanarMuJoCoAdapter:

    def __init__(self, cfg, model, data):
        self.cfg = cfg
        self.model = model
        self.data = data

        self.t = 0.0
        self.lidar_angles = np.linspace(
            -np.pi, np.pi,
            cfg.lidar_n,
            endpoint=False
        )

        self.rng = np.random.default_rng(42)

        self.reset()

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)

        self.x = np.array(
            self.cfg.X0_NOMINAL,
            dtype=float
        )

        self.data.qpos[:3] = self.x
        mujoco.mj_forward(self.model, self.data)

        self.t = 0.0

    def step(self, u):
        v, omega = u

        v = float(np.clip(
            v, -self.cfg.v_max, self.cfg.v_max
        ))

        omega = float(np.clip(
            omega,
            -self.cfg.omega_max,
            self.cfg.omega_max
        ))

        dt = self.cfg.T

        # Differential-drive motion
        theta_mid = self.x[2] + 0.5 * omega * dt

        self.x[0] += v * np.cos(theta_mid) * dt
        self.x[1] += v * np.sin(theta_mid) * dt
        self.x[2] = wrap(self.x[2] + omega * dt)

        # Update MuJoCo robot
        self.data.qpos[:3] = self.x
        mujoco.mj_forward(self.model, self.data)

        self.t += dt

        # Virtual wheel encoder readings
        wl, wr = wheels_from_cmd(v, omega)

        wl += self.rng.normal(0, 0.02)
        wr += self.rng.normal(0, 0.02)

        odom = odom_from_wheels(wl, wr)

        # Sensors not connected yet
        corners = {}
        lidar = np.full(
            self.cfg.lidar_n,
            self.cfg.lidar_max
        )

        return Measurements(
            t=self.t,
            odom=odom,
            corners=corners,
            lidar=lidar
        )

    def ground_truth(self):
        # Evaluation only
        return self.x.copy()

## Geometric LiDAR

In [ ]:
import numpy as np
import mujoco

def simulate_lidar(model, data, cfg, pose, noise_std=0.01):
    """
    Simulate 2D LiDAR against the MuJoCo room's
    circular obstacles and axis-aligned walls.
    """

    x, y, theta = map(float, pose)
    angles = np.linspace(
        -np.pi, np.pi, cfg.lidar_n, endpoint=False
    )

    ranges = np.full(cfg.lidar_n, cfg.lidar_max)

    # Read obstacle locations and sizes from MuJoCo
    obstacles = []

    for i in range(len(cfg.obstacles)):
        gid = mujoco.mj_name2id(
            model, mujoco.mjtObj.mjOBJ_GEOM,
            f"obstacle_{i}"
        )
        if gid < 0:
            raise ValueError(f"Missing obstacle_{i}")

        ox, oy = data.geom_xpos[gid, :2]
        radius = model.geom_size[gid, 0]
        obstacles.append((ox, oy, radius))

    xmin, xmax, ymin, ymax = cfg.room

    for i, angle in enumerate(angles):
        a = theta + angle
        dx, dy = np.cos(a), np.sin(a)

        best = float(cfg.lidar_max)

        # Ray intersections with circular obstacles
        for ox, oy, radius in obstacles:
            fx, fy = x - ox, y - oy
            b = fx * dx + fy * dy
            c = fx**2 + fy**2 - radius**2
            disc = b**2 - c

            if disc >= 0:
                root = np.sqrt(disc)
                hits = [-b - root, -b + root]
                positive = [t for t in hits if t >= 0]
                if positive:
                    best = min(best, min(positive))

        # Ray intersections with room walls
        if abs(dx) > 1e-9:
            for wall_x in (xmin, xmax):
                t = (wall_x - x) / dx
                yy = y + t * dy
                if t >= 0 and ymin <= yy <= ymax:
                    best = min(best, t)

        if abs(dy) > 1e-9:
            for wall_y in (ymin, ymax):
                t = (wall_y - y) / dy
                xx = x + t * dx
                if t >= 0 and xmin <= xx <= xmax:
                    best = min(best, t)

        ranges[i] = best

    # Add small measurement noise
    ranges += np.random.normal(
        0, noise_std, size=len(ranges)
    )

    return np.clip(ranges, 0, cfg.lidar_max)

## LiDAR adapter

In [ ]:
class PlanarMuJoCoWithLiDAR(PlanarMuJoCoAdapter):

    def step(self, u):
        # Run existing motion and encoder simulation
        m = super().step(u)

        # Obtain simulated LiDAR scan
        lidar = simulate_lidar(
            self.model,
            self.data,
            self.cfg,
            self.x
        )

        return Measurements(
            t=m.t,
            odom=m.odom,
            corners=m.corners,
            lidar=lidar
        )

## Geometric camera adapter

In [ ]:
from tb3nav.sim import Sim2D, Measurements
from tb3nav.vision import measurements_from_corners

class PlanarMuJoCoWithCamera(PlanarMuJoCoWithLiDAR):

    def __init__(self, cfg, model, data):
        super().__init__(cfg, model, data)

        # Reuse the camera model from the original ZIP
        self.camera_sim = Sim2D(
            cfg,
            x0=cfg.X0_NOMINAL,
            seed=42,
            wheel_error=0.0
        )

        self.cam = self.camera_sim.cam

    def reset(self):
        super().reset()

        if hasattr(self, "camera_sim"):
            self.camera_sim._x = self.x.copy()
            self.camera_sim.t = self.t

    def step(self, u):
        # Existing robot motion, odometry and LiDAR
        m = super().step(u)

        # Synchronise camera model with MuJoCo robot
        self.camera_sim._x = self.x.copy()
        self.camera_sim.t = self.t

        # Generate projected marker corners
        corners = self.camera_sim._camera()

        return Measurements(
            t=m.t,
            odom=m.odom,
            corners=corners,
            lidar=m.lidar
        )

## Wheel-driven MuJoCo adapter

In [ ]:
import numpy as np
import mujoco

class WheelDrivenMuJoCoAdapter(PlanarMuJoCoWithCamera):

    def __init__(self, cfg, model, data):
        super().__init__(cfg, model, data)

        self.robot_id = mujoco.mj_name2id(
            model, mujoco.mjtObj.mjOBJ_BODY, "robot"
        )
        self.left_id = mujoco.mj_name2id(
            model, mujoco.mjtObj.mjOBJ_ACTUATOR, "left_motor"
        )
        self.right_id = mujoco.mj_name2id(
            model, mujoco.mjtObj.mjOBJ_ACTUATOR, "right_motor"
        )

        self.left_joint = mujoco.mj_name2id(
            model, mujoco.mjtObj.mjOBJ_JOINT, "left_wheel_joint"
        )
        self.right_joint = mujoco.mj_name2id(
            model, mujoco.mjtObj.mjOBJ_JOINT, "right_wheel_joint"
        )

        self.reset()

    def reset(self):
        # Reset physical simulation
        mujoco.mj_resetData(self.model, self.data)
        mujoco.mj_forward(self.model, self.data)

        self.t = 0.0
        self.x = np.array(self.cfg.X0_NOMINAL, dtype=float)

        if hasattr(self, "camera_sim"):
            self.camera_sim._x = self.x.copy()
            self.camera_sim.t = 0.0

    def step(self, u):
        v, omega = map(float, u)

        v = np.clip(v, -self.cfg.v_max, self.cfg.v_max)
        omega = np.clip(
            omega, -self.cfg.omega_max, self.cfg.omega_max
        )

        # Convert robot velocities to wheel velocities
        L = self.cfg.wheel_sep
        r = self.cfg.wheel_radius

        wl = (v - omega * L / 2) / r
        wr = (v + omega * L / 2) / r

        self.data.ctrl[self.left_id] = wl
        self.data.ctrl[self.right_id] = wr

        # Advance MuJoCo physics
        n = round(self.cfg.T / self.model.opt.timestep)

        for _ in range(n):
            mujoco.mj_step(self.model, self.data)

        # Read actual robot pose from MuJoCo
        p = self.data.xpos[self.robot_id]

        R = self.data.xmat[self.robot_id].reshape(3, 3)
        theta = np.arctan2(R[1, 0], R[0, 0])

        self.x = np.array([p[0], p[1], theta])
        self.t = self.data.time

        # Virtual wheel odometry from joint velocities
        li = self.model.jnt_dofadr[self.left_joint]
        ri = self.model.jnt_dofadr[self.right_joint]

        wl_measured = self.data.qvel[li]
        wr_measured = self.data.qvel[ri]

        v_odom = r * (wl_measured + wr_measured) / 2
        w_odom = r * (wr_measured - wl_measured) / L

        odom = (v_odom, w_odom)

        # Existing geometric LiDAR
        lidar = simulate_lidar(
            self.model, self.data, self.cfg, self.x
        )

        # Existing geometric camera
        self.camera_sim._x = self.x.copy()
        self.camera_sim.t = self.t
        corners = self.camera_sim._camera()

        return Measurements(
            t=self.t,
            odom=odom,
            corners=corners,
            lidar=lidar
        )

    def ground_truth(self):
        return self.x.copy()

## Run EKF + LQR autonomous navigation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tb3nav.ekf import EKFConfig, ekf_step
from tb3nav.vision import measurements_from_corners
from tb3nav.control import LQRController
from tb3nav.planner import build_reference
from tb3nav.pipeline import speed_command, lidar_safety

# Reset and initialise robot
sim = WheelDrivenMuJoCoAdapter(
    cfg, model_stable, data_stable
)

# Initialise EKF
ec = EKFConfig(cfg)
xh = np.array(cfg.X0_NOMINAL, dtype=float)
P = np.diag(np.array(cfg.P0_std)**2)

# Initialise LQR
controller = LQRController(cfg)
controller.reset()

# Build reference path
path, clearance = build_reference(
    cfg,
    np.array(cfg.X0_NOMINAL[:2]),
    cfg.X0_NOMINAL[2]
)

# Simulation variables
u = (0.0, 0.0)
idx = 0
hold = 0
reached = False

true_log = []
est_log = []
time_log = []
control_log = []
marker_log = []

# Main navigation loop
for k in range(int(cfg.t_max / cfg.T)):

    # Move robot through MuJoCo wheel actuators
    m = sim.step(u)

    # Simulated camera measurements
    Z = measurements_from_corners(
        cfg, sim.cam, m.corners
    )

    # EKF estimation
    xh, P, nis = ekf_step(
        xh, P, m.odom, Z, ec
    )

    # LQR path tracking
    e_y, e_th, kappa, s, idx = path.errors(
        xh, idx
    )

    d_rem = path.s_end - s

    v = speed_command(cfg, d_rem)
    omega = controller.step(
        e_y, e_th, kappa, v
    )

    # LiDAR safety
    v, omega = lidar_safety(
        cfg,
        sim.lidar_angles,
        m.lidar,
        v,
        omega
    )

    # Stop at destination
    if d_rem < 0.02:
        hold += 1
        v, omega = 0.0, 0.0

        if hold > 10:
            reached = True
    else:
        hold = 0

    u = (v, omega)

    # Ground truth only for evaluation
    true_log.append(sim.ground_truth())
    est_log.append(xh.copy())
    time_log.append(m.t)
    control_log.append(u)
    marker_log.append(len(m.corners))

    if reached:
        break

# Convert to arrays
true_log = np.array(true_log)
est_log = np.array(est_log)
time_log = np.array(time_log)
control_log = np.array(control_log)

# Calculate performance
dock = np.array(cfg.dock_pose)

position_error = np.linalg.norm(
    true_log[-1, :2] - dock[:2]
)

heading_error = (
    true_log[-1, 2] - dock[2] + np.pi
) % (2 * np.pi) - np.pi

estimate_rmse = np.sqrt(
    np.mean(
        np.sum(
            (true_log[:, :2] - est_log[:, :2])**2,
            axis=1
        )
    )
)

print("=== WAFFLE PI NAVIGATION RESULTS ===")
print("Reached path end:", reached)
print("Simulation time:", round(time_log[-1], 2), "s")
print("Final true pose:", true_log[-1])
print("Final estimated pose:", est_log[-1])
print("Docking position error:",
      round(position_error, 4), "m")
print("Docking heading error:",
      round(np.degrees(heading_error), 2), "degrees")
print("EKF position RMSE:",
      round(estimate_rmse, 4), "m")
print("Camera detections:", sum(marker_log))

## Save quantitative evidence and trajectory figure

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("results", exist_ok=True)

# Save numerical trajectory data
np.savez(
    "results/waffle_pi_navigation.npz",
    time=time_log,
    true_pose=true_log,
    estimated_pose=est_log,
    controls=control_log
)

# Save CSV for easy inspection
data = np.column_stack((
    time_log,
    true_log,
    est_log,
    control_log
))

np.savetxt(
    "results/waffle_pi_trajectory.csv",
    data,
    delimiter=",",
    header="time,true_x,true_y,true_theta,ekf_x,ekf_y,ekf_theta,v,omega",
    comments=""
)

# Save performance metrics
metrics = {
    "reached_path_end": bool(reached),
    "simulation_time_s": float(time_log[-1]),
    "docking_position_error_m": float(position_error),
    "docking_heading_error_deg": float(np.degrees(heading_error)),
    "ekf_position_rmse_m": float(estimate_rmse),
    "camera_detections": int(sum(marker_log))
}

with open("results/waffle_pi_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

# Recreate and save the trajectory graph
fig, ax = plt.subplots(figsize=(9, 8))

ax.plot(true_log[:, 0], true_log[:, 1],
        label="Actual trajectory")
ax.plot(est_log[:, 0], est_log[:, 1],
        "--", label="EKF estimated trajectory")

for ox, oy, radius in cfg.obstacles:
    ax.add_patch(
        plt.Circle((ox, oy), radius,
                   color="red", alpha=0.3)
    )

ax.scatter(cfg.X0_NOMINAL[0], cfg.X0_NOMINAL[1],
           label="Start")
ax.scatter(cfg.dock_pose[0], cfg.dock_pose[1],
           marker="*", s=180, label="Dock")

ax.set_title("TurtleBot3 Waffle Pi Navigation")
ax.set_xlabel("X position (m)")
ax.set_ylabel("Y position (m)")
ax.axis("equal")
ax.grid(True)
ax.legend()

fig.savefig(
    "results/waffle_pi_trajectory.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

print("All navigation results saved successfully!")

## Generate 3D replay video (optional; toggle for quicker reruns)

In [ ]:
import mujoco
import numpy as np
import imageio.v2 as imageio
import os
from IPython.display import Video, display

os.makedirs("results", exist_ok=True)

# Create separate data for video replay
replay_data = mujoco.MjData(model_stable)
mujoco.mj_resetData(model_stable, replay_data)
mujoco.mj_forward(model_stable, replay_data)

# Correct Waffle Pi joint indices
x_id = model_stable.jnt_qposadr[mujoco.mj_name2id(model_stable, mujoco.mjtObj.mjOBJ_JOINT, 'x_joint')]
y_id = model_stable.jnt_qposadr[mujoco.mj_name2id(model_stable, mujoco.mjtObj.mjOBJ_JOINT, 'y_joint')]
yaw_id = model_stable.jnt_qposadr[mujoco.mj_name2id(model_stable, mujoco.mjtObj.mjOBJ_JOINT, 'yaw_joint')]

# Initial robot position and heading
robot_id = mujoco.mj_name2id(
    model_stable,
    mujoco.mjtObj.mjOBJ_BODY,
    "robot"
)

initial_x = model_stable.body_pos[robot_id, 0]
initial_y = model_stable.body_pos[robot_id, 1]

# Initial heading from body quaternion
q = model_stable.body_quat[robot_id]
initial_yaw = float(2 * np.arctan2(q[3], q[0]))

print("=== REPLAY CONFIGURATION ===")
print("Initial X:", initial_x)
print("Initial Y:", initial_y)
print("Initial yaw:", initial_yaw)
print("Recorded frames:", len(true_log))

# Camera configuration
camera = mujoco.MjvCamera()
camera.type = mujoco.mjtCamera.mjCAMERA_FREE
camera.lookat[:] = [0.0, 0.0, 0.0]
camera.distance = 7.5
camera.azimuth = 90
camera.elevation = -55

# Create renderer
renderer = mujoco.Renderer(
    model_stable,
    height=480,
    width=640
)

frames = []
fps = 20

sample_times = np.arange(
    time_log[0],
    time_log[-1],
    1.0 / fps
)

for t in sample_times:

    i = min(
        np.searchsorted(time_log, t),
        len(true_log) - 1
    )

    x, y, theta = true_log[i]

    # Convert world coordinates to joint offsets
    replay_data.qpos[x_id] = x - initial_x
    replay_data.qpos[y_id] = y - initial_y
    replay_data.qpos[yaw_id] = theta - initial_yaw

    # Update MuJoCo body positions
    mujoco.mj_forward(model_stable, replay_data)

    # Render frame
    renderer.update_scene(
        replay_data,
        camera=camera
    )

    frames.append(renderer.render().copy())

renderer.close()

# Save MP4
video_path = "results/waffle_pi_3d_navigation_final.mp4"

imageio.mimsave(
    video_path,
    frames,
    fps=fps,
    codec="libx264"
)

print("=== VIDEO SUCCESSFULLY CREATED ===")
print("Saved:", video_path)
print("Frames:", len(frames))
print("Duration:", round(len(frames)/fps, 2), "seconds")

display(Video(video_path, embed=True, width=800))

## Evidence and export

The `results/` directory contains a new trajectory CSV/NPZ, metrics JSON, trajectory PNG and replay MP4. **Run-All is complete when the printed navigation metrics appear and the MP4 is displayed.** The video is a visualization of recorded poses. For the rubric, include a separate explanation of sensor calibration, model controllability/observability, and limitations; this notebook does not establish every criterion by itself.

In [ ]:
# 7. Zip results for downloading; avoid embedding old archived trial results
import zipfile, pathlib
output='TrackA_RunAll_Results.zip'
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path('results').glob('waffle_pi_*'):
        if p.is_file(): z.write(p)
print('Ready:',output)
print('Files:')
with zipfile.ZipFile(output) as z:
    for n in z.namelist(): print(' -',n)